In [ ]:
import random
import math

def sphere(x):
    return sum(xi**2 for xi in x)

def schwefel1(x):
    total = 0
    for i in range(len(x)):
        for j in range(i + 1):
            total += x[j]**2
    return total

def special_forces_algorithm(fitness_func, lower, upper,
                              n_dims=3,
                              population_size=30,
                              iterations=500,
                              tv1=0.5, tv2=0.3,
                              p0=0.25, k=0.1):

    # === INITIALIZATION ===
    soldiers = [
        [random.uniform(lower, upper) for _ in range(n_dims)]
        for _ in range(population_size)
    ]
    fitnesses = [fitness_func(s) for s in soldiers]

    best_idx = min(range(population_size), key=lambda i: fitnesses[i])
    global_best = soldiers[best_idx].copy()
    global_best_fitness = fitnesses[best_idx]

    for t in range(1, iterations + 1):

        # === UNMANNED SEARCH (speed of light mechanism) ===
        # UAV searches randomly around current population knowledge
        # v is a random vector with ||v||^2 = c^2
        c = k * (lower + (1 - t / iterations) * (upper - lower))
        uav_results = []
        for i in range(population_size):
            # generate random v with sum of squares = c^2
            v = [random.gauss(0, 1) for _ in range(n_dims)]
            norm = math.sqrt(sum(vi**2 for vi in v))
            if norm == 0:
                norm = 1e-10
            v = [vi / norm * abs(c) for vi in v]
            xu = [soldiers[i][d] + v[d] for d in range(n_dims)]
            xu = [max(lower, min(upper, xu[d])) for d in range(n_dims)]
            uav_results.append(xu)

        # evaluate UAV positions and update global best if better
        for xu in uav_results:
            f_xu = fitness_func(xu)
            if f_xu < global_best_fitness:
                global_best = xu.copy()
                global_best_fitness = f_xu

        # === INSTRUCTION — command parameter with random factor ===
        instruction = (1 - 0.15 * random.random()) * (1 - t / iterations)

        # === LOSS PROBABILITY — cosine decay ===
        p_loss = p0 * math.cos(math.pi * t / (2 * iterations))

        # === RAID COEFFICIENT — arctan decay ===
        w = 0.75 - 0.55 * math.atan((t / iterations) ** (2 * math.pi))

        # === AVERAGE POSITION (needed for exploitation phase) ===
        x_ave = [
            sum(soldiers[i][d] for i in range(population_size)) / population_size
            for d in range(n_dims)
        ]

        new_soldiers = []

        for i in range(population_size):
            r1 = random.random()
            r2 = random.random()

            if instruction >= tv1:
                # === PHASE 1: EXPLORATION ===

                if r1 >= 0.5:
                    # Large-scale search — wide random jump anchored to global best
                    sign = 1 if random.random() >= 0.5 else -1
                    new_pos = [
                        r1 * (global_best[d] - soldiers[i][d])
                        + sign * (1 - r1) * (upper - lower)
                        for d in range(n_dims)
                    ]
                else:
                    # Raid — move toward best known position
                    # loss probability determines if member knows the true global best
                    if random.random() < p_loss:
                        # lost contact — use a random teammate as aim instead
                        aim_idx = random.randint(0, population_size - 1)
                        x_aim = soldiers[aim_idx]
                        f_aim = fitnesses[aim_idx]
                    else:
                        x_aim = global_best
                        f_aim = global_best_fitness

                    f_i = fitnesses[i]
                    denom = f_i + f_aim
                    if denom == 0:
                        denom = 1e-10
                    ratio = f_i / denom

                    a_i = [ratio * (x_aim[d] - soldiers[i][d]) for d in range(n_dims)]
                    new_pos = [soldiers[i][d] + w * a_i[d] for d in range(n_dims)]

            elif tv2 < instruction < tv1:
                # === PHASE 2: TRANSITION ===

                if r2 >= 0.5:
                    # continue raid behavior
                    if random.random() < p_loss:
                        aim_idx = random.randint(0, population_size - 1)
                        x_aim = soldiers[aim_idx]
                        f_aim = fitnesses[aim_idx]
                    else:
                        x_aim = global_best
                        f_aim = global_best_fitness

                    f_i = fitnesses[i]
                    denom = f_i + f_aim
                    if denom == 0:
                        denom = 1e-10
                    ratio = f_i / denom

                    a_i = [ratio * (x_aim[d] - soldiers[i][d]) for d in range(n_dims)]
                    new_pos = [soldiers[i][d] + w * a_i[d] for d in range(n_dims)]
                else:
                    # move toward global best scaled by instruction + small inertia
                    new_pos = [
                        instruction * (global_best[d] - soldiers[i][d]) + 0.1 * soldiers[i][d]
                        for d in range(n_dims)
                    ]

            else:
                # === PHASE 3: EXPLOITATION (arrest-rescue) ===
                # converge around global best using average position as reference
                r = [random.uniform(-1, 1) for _ in range(n_dims)]
                new_pos = [
                    global_best[d] + r[d] * abs(global_best[d] - x_ave[d])
                    for d in range(n_dims)
                ]

            # clip to bounds
            new_pos = [max(lower, min(upper, new_pos[d])) for d in range(n_dims)]
            new_soldiers.append(new_pos)

        # === UPDATE ===
        soldiers = new_soldiers
        fitnesses = [fitness_func(s) for s in soldiers]

        for i in range(population_size):
            if fitnesses[i] < global_best_fitness:
                global_best = soldiers[i].copy()
                global_best_fitness = fitnesses[i]

        if t % 50 == 0:
            if instruction >= tv1:
                phase = "1-Exploration"
            elif instruction > tv2:
                phase = "2-Transition"
            else:
                phase = "3-Exploitation"
            print(f"Iter {t}: best={global_best_fitness:.4e}, instruction={instruction:.3f}, phase={phase}")

    return global_best, global_best_fitness


print("=== SPHERE ===")
pos, fit = special_forces_algorithm(sphere, -100, 100, n_dims=50,
                                    population_size=30, iterations=500)
print(f"Final best: {fit:.6e}")

print("\n=== SCHWEFEL 1 ===")
pos, fit = special_forces_algorithm(schwefel1, -100, 100, n_dims=50,
                                    population_size=30, iterations=500)
print(f"Final best: {fit:.6e}")